In [37]:
import os
import numpy as np
import pandas as pd

DATA_PATH = '/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/'
baca_df = pd.read_csv(
    os.path.join(DATA_PATH, 'baca_lab_patient_annotations.tsv'),
    sep="\t",
    encoding="utf-8",
    encoding_errors="replace")
text_df = pd.read_csv(os.path.join(DATA_PATH, 'prostate_text_data.csv'))

/tmp/ipykernel_67017/3638050811.py:11: DtypeWarning: Columns (1,5,6,9) have mixed types. Specify dtype option on import or set low_memory=False.
  text_df = pd.read_csv(os.path.join(DATA_PATH, 'prostate_text_data.csv'))


In [38]:
candidate_patients = baca_df.loc[baca_df['Inidcation of Platinum Therapy'].isna(), 
                                 ['DFCI_MRN', 'PLATINUM_CHEMO_MED', 'MEDICATION_START_TIME']]

In [39]:
LLM_text_df = text_df.merge(candidate_patients, on='DFCI_MRN', how='inner')
event_dt = pd.to_datetime(LLM_text_df['EVENT_DATE'], errors='coerce', utc=True)
med_dt = pd.to_datetime(LLM_text_df['MEDICATION_START_TIME'], errors='coerce', utc=True)

LLM_text_df['NOTE_TIME_REL_PLATINUM_START'] = event_dt - med_dt
LLM_text_df['NOTE_TIME_REL_PLATINUM_START'] = LLM_text_df['NOTE_TIME_REL_PLATINUM_START'].apply(lambda x : x.days)

/tmp/ipykernel_67017/2384055891.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  med_dt = pd.to_datetime(LLM_text_df['MEDICATION_START_TIME'], errors='coerce', utc=True)


In [40]:
candidate_LLM_text_df = LLM_text_df.loc[np.abs(LLM_text_df['NOTE_TIME_REL_PLATINUM_START']) <= 120]

In [41]:
candidate_LLM_text_df

,PROC_DESC_STR,PROVIDER_CRED_STR,EVENT_DATE,id,DFCI_MRN,ENCOUNTER_TYPE_DESC_STR,RPT_DATE,RPT_TYPE,SOURCE_STR,PROVIDER_CRED,CLINICAL_TEXT,NOTE_TYPE,PLATINUM_CHEMO_MED,MEDICATION_START_TIME,NOTE_TIME_REL_PLATINUM_START
30,XR CHEST PORTABLE,NaN,2020-06-01T20:11:00Z,68161702601,405085,NaN,2020-06-01T21:26:00Z,Imaging,Imaging,NaN,No acute abnormality. TECHNIQUE XR CHEST PORT...,Imaging,CARBOPLATIN,7/22/20 11:54,-51
31,CT ABDOMEN/PELVIS,NaN,2020-06-01T23:45:00Z,68160227501,708616,NaN,2020-06-02T02:12:00Z,Imaging,Imaging,NaN,Decompressed urinary bladder with a Foley s ...,Imaging,CARBOPLATIN,2/25/20 17:27,97
34,CT FACE,NaN,2020-05-09T20:24:00Z,67516615901,248028,NaN,2020-05-09T21:55:00Z,Imaging,Imaging,NaN,1. Redemonstrated bony destruction of the ante...,Imaging,CARBOPLATIN,5/27/20 16:37,-18
35,XR CHEST PORTABLE,NaN,2020-05-09T19:54:00Z,67516615701,248028,NaN,2020-05-09T20:25:00Z,Imaging,Imaging,NaN,"No acute abnormality. ATTESTATION I, Hei Shun...",Imaging,CARBOPLATIN,5/27/20 16:37,-18
51,XR CHEST PORTABLE,NaN,2019-12-09T20:29:00Z,62122312601,633661,NaN,2019-12-09T22:33:00Z,Imaging,Imaging,NaN,See findings. Findings discussed by Dr Jian Do...,Imaging,CARBOPLATIN,8/21/19 16:29,110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27133,Progress Note,RN,2023-12-04T23:15:00Z,1094793068702,768863,Infusion,NaN,Progress Note,Other,RN,Juan presents to clinic today for Xgeva inject...,Clinician,CARBOPLATIN,12/11/23 14:22,-7
27139,Progress Note,RN,2023-11-28T23:30:00Z,1093526254602,768863,Nurse Only,NaN,Progress Note,Other,RN,Note Subject Protocol 21 086 Cycle 1 Day 1 Mr...,Clinician,CARBOPLATIN,12/11/23 14:22,-13
27144,Progress Note,RN,2024-03-01T02:00:00Z,1146804483502,949252,Infusion,NaN,Progress Note,Other,RN,Don presents to clinic today for labs and bloo...,Clinician,CARBOPLATIN,11/15/23 15:27,106
27151,Progress Note,CNP,2024-03-01T23:30:00Z,1147518538602,567010,Office Visit,NaN,Progress Note,Other,CNP,Date of Service 03/01/24 Medical Oncologist ...,Clinician,CARBOPLATIN,11/15/23 16:29,107
